<a href="https://colab.research.google.com/github/rizkyhaksono/llm-vs-slm-lab/blob/main/02-inference-perbandingan/02_hello_local_slm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02.02 — Hello Local SLM (SmolLM2-135M)

**Tujuan**: load SLM kecil ke memori lokal & generate teks. Bandingkan latency & kualitas dengan LLM via API dari notebook 02.01.

**Prasyarat**: notebook 02.01 lulus.

**Model**: SmolLM2-135M-Instruct (~270 MB di RAM fp32). Kecil banget — sengaja, supaya kontras dengan LLM terasa.

## 0. Bootstrap (jalankan pertama)

In [8]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_NAME = "llm-vs-slm-lab"
    REPO_URL = "https://github.com/rizkyhaksono/llm-vs-slm-lab.git"
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -r requirements.txt

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements.txt").exists():
        repo_root = candidate
        break
assert repo_root is not None
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"IN_COLAB={IN_COLAB}, repo_root={repo_root}")

Cloning into 'llm-vs-slm-lab'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 87 (delta 24), reused 59 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 147.09 KiB | 6.39 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/llm-vs-slm-lab/llm-vs-slm-lab
IN_COLAB=True, repo_root=/content/llm-vs-slm-lab/llm-vs-slm-lab


## 1. Load model + tokenizer

Pertama kali jalan: download ~270MB dari HuggingFace. Selanjutnya pakai cache di `~/.cache/huggingface`.

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model:        {model_id}")
print(f"Total params: {n_params:,} (~{n_params / 1e6:.1f}M)")
print(f"Device:       {next(model.parameters()).device}")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Model:        HuggingFaceTB/SmolLM2-135M-Instruct
Total params: 134,515,008 (~134.5M)
Device:       cpu


## 2. Generate teks (pola aman untuk transformers 4.46+)

Catatan penting: `apply_chat_template(..., return_tensors="pt")` di transformers 4.46+ return **`BatchEncoding`** (dict-like), bukan tensor langsung. Pakai `return_dict=True` eksplisit + `**inputs` untuk unpack.

In [10]:
import time

def generate(prompt: str, max_new_tokens: int = 80, temperature: float = 0.0) -> tuple[str, float]:
    """Generate teks dari SmolLM2. Return (response_text, latency_ms)."""
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    t0 = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature if temperature > 0 else 1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed_ms = (time.perf_counter() - t0) * 1000

    input_len = inputs["input_ids"].shape[1]
    response = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)
    return response, elapsed_ms

# coba 1 prompt
text, ms = generate("Apa ibu kota Indonesia? Jawab singkat.")
print(f"Response: {text}")
print(f"Latency:  {ms:.0f} ms")

Response: Apa ibu kota Indonesia? Jawab singkat.
Latency:  2056 ms


## 3. Bandingkan prompt Bahasa vs English

SmolLM2 di-train dominan corpus English. Jadi prompt English **biasanya lebih natural** outputnya. Demo:

In [11]:
test_pairs = [
    ("What is 2 + 2? Answer in one word.", "Apa hasil 2 + 2? Jawab satu kata."),
    ("List 3 fruits.", "Sebutkan 3 buah."),
    ("What is the capital of France?", "Apa ibu kota Perancis?"),
]

for en, idn in test_pairs:
    text_en, ms_en = generate(en, max_new_tokens=30)
    text_id, ms_id = generate(idn, max_new_tokens=30)
    print(f"[EN]  {en!r}\n  → {text_en[:80].strip()!r}  ({ms_en:.0f} ms)")
    print(f"[IDN] {idn!r}\n  → {text_id[:80].strip()!r}  ({ms_id:.0f} ms)\n")

[EN]  'What is 2 + 2? Answer in one word.'
  → "I'm sorry for the confusion, but as a helpful AI, I don't have the capability to"  (5006 ms)
[IDN] 'Apa hasil 2 + 2? Jawab satu kata.'
  → 'Apa hasil 2 + 2 = 4.'  (2481 ms)

[EN]  'List 3 fruits.'
  → 'Here are three fruits:\n\n1. Banana: A versatile fruit that can be eaten fresh, us'  (3544 ms)
[IDN] 'Sebutkan 3 buah.'
  → 'Sebetkan 3 buah.'  (1269 ms)

[EN]  'What is the capital of France?'
  → 'The capital of France is Paris.'  (1060 ms)
[IDN] 'Apa ibu kota Perancis?'
  → 'Apa ibu kota Perancis?'  (1450 ms)



## 4. Latency 3x — ambil median

Latency di CPU laptop noisy karena thermal/throttling. Sampling 3x, ambil median, lebih stabil.

In [12]:
from statistics import median

prompt = "Tuliskan 3 manfaat olahraga."
runs = []
for i in range(3):
    _, ms = generate(prompt, max_new_tokens=60)
    runs.append(ms)
    print(f"Run {i+1}: {ms:.0f} ms")

print(f"\nMin:    {min(runs):.0f} ms")
print(f"Median: {median(runs):.0f} ms")
print(f"Max:    {max(runs):.0f} ms")

Run 1: 2605 ms
Run 2: 3404 ms
Run 3: 1938 ms

Min:    1938 ms
Median: 2605 ms
Max:    3404 ms


## 5. Bandingkan dengan LLM via Groq

Prompt yang sama, di-eksekusi di:
1. SmolLM2-135M lokal CPU (model 135M params)
2. Llama-3.1-8B di Groq (model **60x lebih besar**, hardware khusus LPU)

In [14]:
from utils.llm_clients import GROQ_DEFAULT_MODEL, groq_client

client = groq_client()

prompt = "Tuliskan 3 manfaat olahraga, satu kalimat per poin, dalam Bahasa Indonesia."

# SLM lokal
text_slm, ms_slm = generate(prompt, max_new_tokens=120)

# LLM via Groq
t0 = time.perf_counter()
r = client.chat.completions.create(
    model=GROQ_DEFAULT_MODEL,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=120,
    temperature=0,
)
ms_llm = (time.perf_counter() - t0) * 1000
text_llm = r.choices[0].message.content

print("=" * 70)
print(f"SLM (SmolLM2-135M lokal, {ms_slm:.0f} ms):")
print(text_slm)
print("=" * 70)
print(f"LLM (Groq llama-3.1-8b, {ms_llm:.0f} ms):")
print(text_llm)
print("=" * 70)
print(f"\nRatio latency: {ms_slm / ms_llm:.1f}x lebih cepat di Groq")

SLM (SmolLM2-135M lokal, 2672 ms):
Tuliskan 3 manfaat olahraga, satu kalimat per poin, dalam Bahasa Indonesia.
LLM (Groq llama-3.1-8b, 1590 ms):
Berikut 3 manfaat olahraga:

1. Olahraga dapat meningkatkan kesehatan fisik dan mental, sehingga meningkatkan kualitas hidup.
2. Olahraga dapat membantu mengurangi stres dan kecemasan, sehingga meningkatkan kualitas tidur dan meningkatkan mood.
3. Olahraga dapat meningkatkan kekuatan otot dan meningkatkan kemampuan fisik, sehingga meningkatkan kemampuan dalam melakukan aktivitas sehari-hari.

Ratio latency: 1.7x lebih cepat di Groq


## Refleksi & insight

1. **SmolLM2-135M di CPU itu lambat** (biasanya 1–5 detik per 50 token). Itu *fitur*: kamu sekarang punya feel kenapa orang bayar API LLM.
2. **Kualitas SLM kecil untuk Bahasa Indonesia terbatas** — sering ngalor-ngidul, ngulang-ulang, atau switch ke English. Karena corpus training dominan English.
3. **Tradeoff jelas**: SLM lokal = gratis + privat + offline-able, tapi quality terbatas + slow.
4. **Solusi quantization** (notebook berikut) bisa nambah ukuran efektif (TinyLlama 1.1B) tanpa OOM, dengan latency yang lebih manageable.
5. Untuk task **klasifikasi/extraction** (bukan generative), SLM kecil bisa **mengalahkan LLM** kalau di-fine-tune. Itu cerita modul 03.

## Latihan mandiri

1. Coba ganti model_id ke `"HuggingFaceTB/SmolLM2-360M-Instruct"` (2.6x lebih besar). Berapa lipat lebih lambat? Apakah kualitas Bahasa-nya lebih baik?
2. Coba parameter `temperature=0.8` di generate(). Bandingkan dengan greedy (`temperature=0.0`) — output mana yang lebih natural untuk task kreatif?

## Lanjut

Sekarang naikkan ke SLM 1.1B (Q4 quantized) yang masih masuk akal di CPU: [03_quantization_dengan_llamacpp.ipynb](03_quantization_dengan_llamacpp.ipynb)